In [1]:
import numpy as np
import cupy as cp
import igpu, pickle

In [2]:
def to_day(di):
    if di >= 2:
        return "day6"
    else:
        return "day" + str(di + 1)

In [3]:
w = []
h = []
for mi in range(6):
    ws = []
    hs = []
    for di in range(3):
        mouse = "mouse" + str(mi + 1)
        day = to_day(di)
        x = igpu.open_csv_nonull(mouse + "/" + mouse + "_" + day + ".csv")
        w_save = None
        h_save = None
        cost_min = np.inf
        for i in range(10):
            model = igpu.solve(x.T, 20, 50, z_th=x.min(), random_seed=i*100+mi*10+di*2, corr_max=0.3, comp_rate=0.1, Wlim=0.01, Hlim=1.0)
            c = model.reconstruction_cost()
            updated = ""
            if c < cost_min:
                cost_min = c
                updated = "updated"
                w_save = cp.asnumpy(model.W)
                h_save = cp.asnumpy(model.H.T)
            print(mouse, day, "| iter = {:>2} | k = {:>2}, cost = {:.3f}".format(i + 1, model.k, c), updated)
        ws.append(w_save)
        hs.append(h_save)
    w.append(ws)
    h.append(hs)

mouse1 day1 | iter =  1 | k =  4, cost = 25741002.509 updated
mouse1 day1 | iter =  2 | k =  3, cost = 26272607.533 
mouse1 day1 | iter =  3 | k =  5, cost = 25848701.467 
mouse1 day1 | iter =  4 | k =  5, cost = 25484586.951 updated
mouse1 day1 | iter =  5 | k =  3, cost = 26148361.076 
mouse1 day1 | iter =  6 | k =  3, cost = 26653402.857 
mouse1 day1 | iter =  7 | k =  3, cost = 26636365.665 
mouse1 day1 | iter =  8 | k =  4, cost = 26250769.778 
mouse1 day1 | iter =  9 | k =  6, cost = 25272434.317 updated
mouse1 day1 | iter = 10 | k =  4, cost = 25795531.553 
mouse1 day2 | iter =  1 | k =  3, cost = 18052541.039 updated
mouse1 day2 | iter =  2 | k =  3, cost = 18012909.960 updated
mouse1 day2 | iter =  3 | k =  3, cost = 18362909.532 
mouse1 day2 | iter =  4 | k =  3, cost = 18277327.205 
mouse1 day2 | iter =  5 | k =  3, cost = 18393952.527 
mouse1 day2 | iter =  6 | k =  4, cost = 17796492.989 updated
mouse1 day2 | iter =  7 | k =  4, cost = 18087204.109 
mouse1 day2 | iter =  8

In [4]:
with open("w.pkl", "wb") as f:
    pickle.dump(w, f)
with open("h.pkl", "wb") as f:
    pickle.dump(h, f)